# Cosine Similarity

In this section you will construct another similarity metric, now based on the cosinus.

Remember trigonometry (or better, linear algebra!) from your mathematics class? Well this metric is based on trigonometric operations and calculates the angle between two vectors. It might look difficult but it is rather simple. 

## 1. Load the dataset

In [1]:
# code goes here
import pandas as pd 
df = pd.read_csv('data/BX-Book-Ratings-Filtered.csv', sep=';', encoding='latin-1')
df

,User-ID,ISBN,Book-Rating
0,4017,006000438X,10
1,4017,0060915544,8
2,4017,0060929871,9
3,4017,0060930187,10
4,4017,0060964049,9
...,...,...,...
11360,276050,0553377868,7
11361,276050,0671021001,9
11362,276050,067102423X,8
11363,276050,0679746048,7


## 2. Explore the dimensions (shape) of users' rating data

Here are the IDs of two users in our ratings dataset. What are their respective ratings' dimensions (shape)? How many books did these users rate respectively?

In [2]:
user_id_a = 277427
user_id_b = 277203

# code goes here
ratings_a = df[df["User-ID"] == user_id_a]
ratings_b = df[df["User-ID"] == user_id_b]

ratings_a.shape, ratings_b.shape

((0, 3), (0, 3))

If we are to produce vectors from the users' ratings and apply trigonometric operations on them,  can you see a problem here? Are the vectors of the same dimension? If not, why is this a 'problem'?

## 3. Vectorize ratings

Can you vectorize the above users' ratings so they have the same dimension? To help you do this, here is sorted  list of all the ISBNs in our dataset. How can you use this list of all the ISBNs to create a (large!) vector for user_id_a?

In [6]:
import numpy as np

ISBNS_array = df['ISBN'].unique()
ISBNS_array = np.sort(ISBNS_array).tolist()
# Create vector representations for users' ratings
def vectorize_ratings(user_id, isbn_list, df):
    """
    Create a vector representation of a user's ratings.
    The vector will have the same length as the total number of unique ISBNs.
    If a user has rated a book, the rating is inserted; otherwise, it's set to 0.
    """
    user_ratings = df[df["User-ID"] == user_id].set_index("ISBN")["Book-Rating"].to_dict()
    return np.array([user_ratings.get(isbn, 0) for isbn in isbn_list])

# Vectorizing ratings for example users
user_id_a = 4017  
user_id_b = 6242  

user_a_vector = vectorize_ratings(user_id_a, ISBNS_array, df)
user_b_vector = vectorize_ratings(user_id_b, ISBNS_array, df)

# Display the first few elements of the vectors
user_a_vector[:10], user_b_vector[:10]


(array([ 0, 10,  0,  0,  0,  0,  0,  0,  0,  0]),
 array([0, 6, 0, 0, 0, 0, 0, 0, 0, 0]))

## 4. Helper functions

Below are two functions that (1) retrieve user ratings from a given dataset and (2) vectorize these ratings according to a certain dimension (all ISBNs).

In [4]:
def get_user_ratings(user_id, df_subset):
    
    df_user = df_subset[df_subset['User-ID'] == user_id]
        
    return dict(zip(df_user['ISBN'], df_user['Book-Rating']))

In [5]:
def create_ratings_vector(user_ISBN_rating_dict, all_ISBNS_array):
    
    user_ISBNS = user_ISBN_rating_dict.keys()
    
    return [0 if v not in user_ISBNS else user_ISBN_rating_dict[v] for v in all_ISBNS_array]    

## 5. Cosine distance function

Can you finish the writing of the function below that calculates the angle between two vectors have the same dimension? As you can see, use the numpy `dot` and `norm` operators do translate the given formula into code.

In [7]:
# Import necessary functions
from numpy import dot
from numpy.linalg import norm

def cosine_distance(ratings_vector_user_a, ratings_vector_user_b):
    """
    Computes the cosine distance (1 - cosine similarity) between two rating vectors.
    """
    denominator = norm(ratings_vector_user_a) * norm(ratings_vector_user_b)
    if denominator == 0:
        return 0
    
    return 1 - (dot(ratings_vector_user_a, ratings_vector_user_b) / denominator)

cosine_dist = cosine_distance(user_a_vector, user_b_vector)
cosine_dist


np.float64(0.7596892956414876)

## 6. Calculate distances

Here is the ID of a user in our dataset (you can of course choose another one!).

Can you calculate this user's cosine distance from all the other users in the dataset?

In [12]:
a_user_id = 276050

# Define a function to calculate cosine distances for a given user ID
def calculate_user_distances(target_user_id, df, isbn_list):
    """
    Computes cosine distances between a given user and all other users in the dataset.
    
    Parameters:
    - target_user_id: The user ID for whom to compute distances.
    - df: The ratings dataframe.
    - isbn_list: List of all unique ISBNs to ensure consistent vectorization.
    
    Returns:
    - A sorted DataFrame of cosine distances (lower means more similar).
    """
    # Vectorize the target user's ratings
    target_vector = vectorize_ratings(target_user_id, isbn_list, df)
    
    # Compute distances for all other users
    user_distances = {
        other_user: cosine_distance(target_vector, vectorize_ratings(other_user, isbn_list, df))
        for other_user in df["User-ID"].unique() if other_user != target_user_id
    }
    
    # Convert to DataFrame and sort by distance
    user_distances_df = pd.DataFrame.from_dict(user_distances, orient='index', columns=['Cosine Distance'])
    user_distances_df.sort_values(by="Cosine Distance", ascending=True, inplace=True)  # Lower is more similar
    
    return user_distances_df

user_distances_result = calculate_user_distances(a_user_id, df, ISBNS_array)

user_distances_result


,Cosine Distance
138543,0.816958
159506,0.820380
181687,0.829100
162738,0.829676
242299,0.849904
...,...
68555,1.000000
274004,1.000000
6251,1.000000
270554,1.000000


## 7. Function calculating distances

Considering the code above, can you make a function that will take as input a given user's ID and calculate its distance from all other users in our dataset?

In [ ]:
# code goes here